In [1]:
# Vi importerar det aggregerade datasetet från Morrins studie
import pandas as pd
from math import floor

def read_and_filter_data(year_filter):
    df = pd.read_csv(r".\File_9_LifeExpectancy_DecilesIncome_IndividualIncome.csv")

    # Filtrerar
    if year_filter > 0:
        df = df[df['year'] == year_filter]

    return df

In [2]:
# Fristående lönesimulering, ett 10 000 lognormala löner har simulerats med en fördelning peggad mot siffror från pensionsmyndigheten, sedan har fördelningen
# delats in i 10 lika stor grupper, där gruppmedel anges nedan

def get_v0_df():
    kvinnor_v0 = {  "1":780997,
                    "2":1125171,
                    "3":1392136,
                    "4":1650803,
                    "5":1921606,
                    "6":2244620,
                    "7":2635908,
                    "8":3132064,
                    "9":3865826,
                    "10":5502823}

    man_v0 = {  "1":875172,
                "2":1260112,
                "3":1570483,
                "4":1888806,
                "5":2213281,
                "6":2595123,
                "7":3071718,
                "8":3685956,
                "9":4655568,
                "10":6855691}

    df_temp_k = pd.DataFrame([kvinnor_v0.keys(), kvinnor_v0.values()]).T
    df_temp_k["sex"] = 2

    df_temp_m = pd.DataFrame([man_v0.keys(), man_v0.values()]).T
    df_temp_m["sex"] = 1

    df_lon = pd.concat([df_temp_k, df_temp_m]).reset_index().iloc[:, 1:]
    df_lon.columns = ["level_income", "v0", "sex"]
    df_lon["level_income"] = df_lon["level_income"].astype("Int64")

    return df_lon




In [3]:
# Vi grupperar enligt våra 20 typfall och joinar in pensionsbehållningen

def combine_tables(df, df_lon, pensionsalder):
    df_sex_income_yl = df[df['age'] == pensionsalder].groupby(["sex", "level_income"]).mean().reset_index()[["sex", "level_income", "_ExpYL"]].copy(deep=True)
    df_sex_income_yl["overall_mean"] = df_sex_income_yl["_ExpYL"].mean()
    df_final = df_sex_income_yl.merge(df_lon, how="left", on = ["sex", "level_income"])

    return df_final

In [4]:
# Tabell för att demonstrera beståndsstatistiken
def generate_descriptive_table_1(df):
    df_table1 = df[(df['age'] == 65) & (df['sex'] == 1)].groupby(["sex", "level_income", "year"]).mean().reset_index()[["sex", "level_income", "year", "_ExpYL"]].copy(deep=True)
    df_table1 = df_table1[df_table1['year'].isin([2006, 2008, 2010, 2012, 2014])]
    df_table1 = df_table1.pivot(index = ['sex', 'level_income'], columns='year').reset_index()
    return df_table1

In [5]:
def berakna_delningstal(df, income_filter, sex_filter, franta, pensionsalder):
    df_filtered = df.copy()
    i = pensionsalder

    if income_filter > 0:
        df_filtered = df_filtered.copy()[df_filtered["level_income"] == income_filter]
    if sex_filter > 0:
        df_filtered = df_filtered.copy()[df_filtered["sex"] == sex_filter]

    temp = df_filtered.groupby(["age"]).mean().reset_index()[["age", "_lx"]]

    ages = min(len(temp[~temp["_lx"].isna()]), 45)
    L = {temp["age"][j] : temp["_lx"][j] for j in range(ages)}

    scaler = 1 / (12 * L[i])

    Di = 0

    for k in range (i, i + ages -5):
        for X in range(0, 12):
            val1 = (L[k] + (L[k+1] - L[k]) * X / 12)
            val2 = franta**-(k-i) * franta**-(X/12)
            Di += scaler * val1 * val2


    return Di 


In [6]:
def berakna_utfall(df_final, formel_delningstal, avk_ranta, franta):

    df_final = df_final.copy()

    # Bestämmer faktisk återstående livslängd med förskottsränta 0 % (alltså franta = 1.0)
    df_final['ExpYL'] = df_final.apply(lambda x: formel_delningstal(x.level_income, x.sex, franta = 1.0, v0 = 0), axis=1)

    # Bestämmer delningstal 
    df_final['delningstal'] = df_final.apply(lambda x: formel_delningstal(0, 0, franta, x.v0), axis=1)
    
    df_final['startbelopp'] = df_final['v0'] / df_final['delningstal'] / 12

    if avk_ranta != franta:
        df_final['totalt_utbetalt'] = 12 * df_final['startbelopp'] * ((1 + avk_ranta - franta)**( df_final['ExpYL'].apply(floor)) - 1) / (avk_ranta - franta) + (df_final['ExpYL'] - df_final['ExpYL'].apply(floor))*(1 + avk_ranta - franta)**df_final['ExpYL']
    else:
        df_final['totaltt_utbetalt'] = df_final['v0']

    # Justerar totalt utbetalt
    totalt_utbetalt_alla = df_final['totalt_utbetalt'].sum()
    totalt_v0_alla = df_final['v0'].sum()
    df_final['totalt_utbetalt'] = df_final['totalt_utbetalt'] * (totalt_v0_alla / totalt_utbetalt_alla)


    df_final['erhallen_procent'] = df_final['totalt_utbetalt'] / df_final['v0']

    return df_final

In [7]:
# Parametrar
franta = 1.016
avk_ranta = 1.024
pensionsalder = 65

# Data prep
df = read_and_filter_data(year_filter=0)
df_lon = get_v0_df()
df_final = combine_tables(df, df_lon, pensionsalder)

kapitalvikt = 1 / df_final['v0'].std()

formel_dagens = lambda level_income, sex, franta, v0 : berakna_delningstal(df, level_income, sex, franta, pensionsalder) 
formel_kapitalviktat = lambda level_income, sex, franta, v0 : formel_dagens(level_income, sex, franta, avk_ranta) + kapitalvikt * v0

df_dagens = berakna_utfall(df_final, formel_dagens, avk_ranta = avk_ranta, franta = franta)
df_kapitalviktat = berakna_utfall(df_final, formel_kapitalviktat, avk_ranta = avk_ranta, franta = franta)


# Vi antar konstant utveckling av löneindex, 0.8, 1.6 eller 2.4 eller 3.2
# Totalt utbetalt belopp bör vara oförändrat, kan vi normalisera och täta på något sätt? 


In [ ]:
# Nästa steg att smattra på med tabeller att skissa upp i Excel
df_kapitalviktat

,sex,level_income,_ExpYL,overall_mean,v0,ExpYL,delningstal,startbelopp,totalt_utbetalt,erhallen_procent
0,1,1,16.380604,19.860024,875172,16.428258,17.232782,4232.10826,756658.14823,0.864582
1,1,2,16.637789,19.860024,1260112,16.686364,17.472669,6009.919315,1074512.861398,0.852712
2,1,3,17.542553,19.860024,1570483,17.589449,17.666087,7408.18191,1413030.79858,0.899743
3,1,4,17.986077,19.860024,1888806,18.031434,17.864459,8810.817989,1786693.649081,0.945938
4,1,5,18.273008,19.860024,2213281,18.313483,18.066666,10208.861117,2070194.824352,0.935351
5,1,6,18.657151,19.860024,2595123,18.699774,18.304622,11814.515707,2395796.406788,0.923192
6,1,7,19.032392,19.860024,3071718,19.062787,18.601627,13760.973479,2957582.582801,0.962843
7,1,8,19.577946,19.860024,3685956,19.609655,18.984409,16179.750812,3477439.792176,0.94343
8,1,9,20.222358,19.860024,4655568,20.264523,19.588652,19805.548582,4499104.34771,0.966392
9,1,10,21.104795,19.860024,6855691,21.141838,20.959726,27257.397182,6528151.215487,0.952224
